In [ ]:
# The mechanistic justification for this step is: [Environment Setup]
# Configure Kaggle pathing invariant and fallback paths, initialize PyTorch framework.

import os
import torch
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"CUDA memory allocated: {torch.cuda.memory_allocated() if torch.cuda.is_available() else 0}")

# Kaggle Pathing Invariant
INPUT_DIR = '/kaggle/input/afnd-splits' if os.path.exists('/kaggle/input/afnd-splits') else 'data/splits'
OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else 'experiments'
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# The mechanistic justification for this step is: [Data Loading]
# Load the random splits to evaluate Source-Induced Credibility (Task A).

print(f"Loading random splits from {INPUT_DIR}...")
train_df = pd.read_parquet(f'{INPUT_DIR}/train_rand.parquet')
val_df = pd.read_parquet(f'{INPUT_DIR}/val_rand.parquet')
test_df = pd.read_parquet(f'{INPUT_DIR}/test_rand.parquet')

label_map = {l: i for i, l in enumerate(train_df['label'].unique())}
for df in [train_df, val_df, test_df]:
    df['label'] = df['label'].map(label_map)

# Using full dataset or high sampling for robust extended training
train_df = train_df.sample(n=min(len(train_df), 200000), random_state=42)
print(f"Sampled Train: {len(train_df)}, Val: {len(val_df)}")


In [ ]:
# The mechanistic justification for this step is: [Model & Dataset Initialization]
# Embed necessary dataset and model class definitions so the Kaggle notebook runs standalone, without external src/ dependencies.

import torch.nn as nn
from transformers import AutoModel

class TransformerDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super(TransformerClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        return self.classifier(output)

MAX_LEN = 128
BATCH_SIZE = 16  # Kaggle T4x2 can handle slightly larger batches
model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_trans_ds = TransformerDataset(train_df['title'] + " " + train_df['text'], train_df['label'], tokenizer, max_len=MAX_LEN)
val_trans_ds = TransformerDataset(val_df['title'] + " " + val_df['text'], val_df['label'], tokenizer, max_len=MAX_LEN)

train_trans_loader = DataLoader(train_trans_ds, batch_size=BATCH_SIZE, shuffle=True) 
val_trans_loader = DataLoader(val_trans_ds, batch_size=BATCH_SIZE, shuffle=False)

arabert = TransformerClassifier(model_name=model_name, num_classes=len(label_map)).to(device)


In [ ]:
# The mechanistic justification for this step is: [Extended Training Loop]
# Execute a 10-epoch training loop to maximize model convergence, saving the best checkpoint based on validation loss to /kaggle/working.

optimizer_arabert = torch.optim.AdamW(arabert.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

epochs = 10
best_val_loss = float('inf')

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    arabert.train()
    total_train_loss = 0
    for batch_idx, batch in enumerate(train_trans_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer_arabert.zero_grad()
        outputs = arabert(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer_arabert.step()
        total_train_loss += loss.item()
        
        if batch_idx % 1000 == 0:
            print(f"Batch {batch_idx}/{len(train_trans_loader)} | Loss: {loss.item():.4f}")
    
    avg_train_loss = total_train_loss / len(train_trans_loader)
    print(f"Avg Train Loss: {avg_train_loss:.4f}")
    
    arabert.eval()
    total_val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in val_trans_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = arabert(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
    avg_val_loss = total_val_loss / len(val_trans_loader)
    val_acc = correct / total
    print(f"Avg Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_acc:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(arabert.state_dict(), f"{OUTPUT_DIR}/arabert_best_10epochs.pt")
        print(f"Saved new best model to {OUTPUT_DIR}/arabert_best_10epochs.pt with Val Loss: {best_val_loss:.4f}")
